[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/part_05_classical_ml/ch21_ml_paradigm_leakage/01_walkthrough.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `INTERACTIVE STUDENT NOTEBOOK`


# Chapter 21 Walkthrough: The Machine Learning Paradigm — Framing Problems, Data Leakage, Splits, and Bias-Variance Tradeoff

**Book Reference:** Chapter 21 (Opening Part V: Classical Machine Learning), Sections 21.1 – 21.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every experiment uses the bundled `mch_survey_sample` and `epi_surveillance` research datasets.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & Research Datasets
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
epi_raw = tutor.load_data("epi_surveillance", as_pandas=True)

mch = mch_raw.copy()
mch["residence"] = mch["residence"].astype(str).str.strip().str.title()
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)
w_map = {"Poorest": 1, "Poorer": 2, "Middle": 3, "Richer": 4, "Richest": 5}
mch["wealth_idx"] = mch["wealth_quintile"].map(w_map)

feat_cols = [
    "maternal_age_years", "anc_visits", "is_urban",
    "wealth_idx", "child_haz",
]
keep_cols = ["household_id", "cluster_id"] + feat_cols + ["maternal_anemia"]
clean_mch = mch[keep_cols].dropna().reset_index(drop=True)
print("clean_mch shape:", clean_mch.shape)
print("epi_surveillance shape:", epi_raw.shape)


## Section 21.3.1 — Prediction vs. Explanatory Inference (Framing X and y)

In [ ]:
X = clean_mch[feat_cols].copy()
y = clean_mch["maternal_anemia"].copy()

print("Feature matrix X shape:", X.shape)
print("Target y class counts:", y.value_counts().to_dict())
print(f"Overall anemia prevalence: {y.mean():.4f}")


## Section 21.3.2 — Stratified 3-Way Train / Validation / Test Split (60 / 20 / 20)

In [ ]:
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
val_rel = 0.20 / (1.0 - 0.20)  # 0.25 of remaining 80% = 20% of total
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=val_rel, stratify=y_tmp, random_state=42
)

print(f"Rows -> train={len(X_train)}, val={len(X_val)}, "
      f"test={len(X_test)} (total={len(X_train)+len(X_val)+len(X_test)})")
print(f"Positive rate -> train={y_train.mean():.4f}, "
      f"val={y_val.mean():.4f}, test={y_test.mean():.4f}")


## Section 21.3.3 — Preventing DHS Cluster Leakage with `GroupShuffleSplit`

In [ ]:
# Naive random split leaks DHS clusters across train and test
tr_rand, te_rand = train_test_split(
    clean_mch, test_size=0.20,
    stratify=clean_mch["maternal_anemia"], random_state=42,
)
rand_overlap = len(set(tr_rand["cluster_id"]) & set(te_rand["cluster_id"]))
print(f"Naive random split -> {rand_overlap} of "
      f"{te_rand['cluster_id'].nunique()} test clusters leaked!")

# Proper cluster-aware split using GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_idx, te_idx = next(
    gss.split(clean_mch, groups=clean_mch["cluster_id"])
)
tr_grp = clean_mch.iloc[tr_idx].reset_index(drop=True).copy()
te_grp = clean_mch.iloc[te_idx].reset_index(drop=True).copy()
grp_overlap = len(set(tr_grp["cluster_id"]) & set(te_grp["cluster_id"]))
n_tr_c = tr_grp["cluster_id"].nunique()
n_te_c = te_grp["cluster_id"].nunique()
print(f"GroupShuffleSplit -> train={len(tr_grp)} ({n_tr_c} clusters), "
      f"test={len(te_grp)} ({n_te_c} clusters)")
print(f"Cluster overlap: {grp_overlap}")


## Section 21.3.4 — Chronological Temporal Split on Outbreak Surveillance

In [ ]:
epi = epi_raw.copy()
epi_train = epi.loc[epi["year"] < 2025].reset_index(drop=True).copy()
epi_test = epi.loc[epi["year"] >= 2025].reset_index(drop=True).copy()

print(f"Train weeks: {len(epi_train)} (max year={epi_train['year'].max()})")
print(f"Test weeks:  {len(epi_test)} (min year={epi_test['year'].min()})")
has_time_leak = bool(epi_train["year"].max() >= epi_test["year"].min())
print("Temporal leak detected:", has_time_leak)


## Section 21.3.5 — Live Preprocessing Leakage vs. Fit-on-Train-Only Scaling

In [ ]:
# 1. High-dimensional cohort with uneven feature scales
X_hd, y_hd = make_classification(
    n_samples=60, n_features=30, n_informative=5,
    n_redundant=5, random_state=2,
)
X_hd = X_hd * np.linspace(0.1, 100.0, 30)

# LEAKY: fit StandardScaler on ALL rows before splitting
X_all_s = StandardScaler().fit_transform(X_hd)
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(
    X_all_s, y_hd, test_size=0.20, stratify=y_hd, random_state=42
)
leaky_acc = LogisticRegression(max_iter=500).fit(Xl_tr, yl_tr).score(
    Xl_te, yl_te
)

# CLEAN: split FIRST, fit StandardScaler on train ONLY
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    X_hd, y_hd, test_size=0.20, stratify=y_hd, random_state=42
)
scaler = StandardScaler().fit(Xc_tr)
clean_acc = LogisticRegression(max_iter=500).fit(
    scaler.transform(Xc_tr), yc_tr
).score(scaler.transform(Xc_te), yc_te)

print(f"Leaky accuracy: {leaky_acc:.4f} | Clean accuracy: {clean_acc:.4f}")
print(f"Phantom optimism gap: {leaky_acc - clean_acc:+.4f}")

# 2. Verify train-only scaling on MCH features
mch_scaler = StandardScaler()
X_tr_s = mch_scaler.fit_transform(X_train)
X_te_s = mch_scaler.transform(X_test)
print("MCH train scaled means:", np.round(X_tr_s.mean(axis=0), 3))
print("MCH test scaled means: ", np.round(X_te_s.mean(axis=0), 3))


## Section 21.3.6 — Zero-Skill Baseline (`DummyClassifier`)

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
maj_class = int(dummy.predict(X_test[:1])[0])
dummy_acc = float(dummy.score(X_test, y_test))
pos_rate = float(np.mean(y_test == 1))

print(f"Majority class predicted: {maj_class}")
print(f"Dummy accuracy floor:     {dummy_acc:.4f}")
print(f"Test positive rate:       {pos_rate:.4f}")


## Section 21.3.7 — Bias-Variance Tradeoff & Learning Curve Diagnostics

In [ ]:
# 1. Complexity sweep: DecisionTree max_depth from 1 to None
for depth in [1, 3, 6, 12, None]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    tr_s = dt.score(X_train, y_train)
    va_s = dt.score(X_val, y_val)
    print(f"depth={str(depth):>4} | train={tr_s:.4f} | "
          f"val={va_s:.4f} | gap={tr_s - va_s:+.4f}")

# 2. Learning curve: vary training sample size on 75/25 stratified split
X_lc_tr, X_lc_val, y_lc_tr, y_lc_val = train_test_split(
    X.to_numpy(), y.to_numpy(), test_size=0.25,
    stratify=y, random_state=42,
)
lc_rows = []
for frac in [0.2, 0.4, 0.6, 0.8, 1.0]:
    n_sub = int(round(frac * len(X_lc_tr)))
    clf = DecisionTreeClassifier(max_depth=4, random_state=42)
    clf.fit(X_lc_tr[:n_sub], y_lc_tr[:n_sub])
    tr_a = float(clf.score(X_lc_tr[:n_sub], y_lc_tr[:n_sub]))
    va_a = float(clf.score(X_lc_val, y_lc_val))
    lc_rows.append({
        "fraction": frac, "n_train": n_sub,
        "train_acc": round(tr_a, 4), "val_acc": round(va_a, 4),
        "gap": round(tr_a - va_a, 4),
    })
print("\nLearning Curve Table (max_depth=4):")
print(pd.DataFrame(lc_rows).to_string(index=False))


## Section 21.3.8 — Break It On Purpose! (ML Splitting & Scaling Error Anatomy)

In [ ]:
# Intentional Bug #1: ValueError — stratifying a class with only 1 sample
X_tiny = np.arange(10).reshape(-1, 1)
y_singleton = [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
train_test_split(X_tiny, y_singleton, test_size=0.2, stratify=y_singleton)


In [ ]:
# Intentional Bug #2: ValueError — mismatched X and y lengths in split
X_ten = np.arange(10).reshape(-1, 1)
y_five = [0, 1, 0, 1, 0]
train_test_split(X_ten, y_five, test_size=0.2)


In [ ]:
# Intentional Bug #3: NotFittedError — calling .transform() before .fit()
unfitted_scaler = StandardScaler()
unfitted_scaler.transform([[25.0, 4.0, 1.0]])


## Section 21.4 — Stage 4 Blueprint: Leak-Free DHS Cluster Modeling Benchmark

In [ ]:
def run_leak_free_mch_benchmark(raw_df, seed=42):
    """Cluster-split, train-only scale, and benchmark 3 ML models."""
    df = raw_df.copy()
    df["residence"] = df["residence"].astype(str).str.strip().str.title()
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3, "Richer": 4, "Richest": 5,
    }
    df["wealth_idx"] = df["wealth_quintile"].map(w_map)
    feats = [
        "maternal_age_years", "anc_visits", "is_urban",
        "wealth_idx", "child_haz",
    ]
    cols = ["household_id", "cluster_id"] + feats + ["maternal_anemia"]
    clean = df[cols].dropna().reset_index(drop=True)

    # 1. DHS cluster-aware split + zero-leakage assertion
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    tr_i, te_i = next(gss.split(clean, groups=clean["cluster_id"]))
    tr_df = clean.iloc[tr_i].reset_index(drop=True)
    te_df = clean.iloc[te_i].reset_index(drop=True)
    assert len(set(tr_df["cluster_id"]) & set(te_df["cluster_id"])) == 0

    # 2. Fit scaler strictly on training clusters
    scaler = StandardScaler().fit(tr_df[feats])
    X_tr = scaler.transform(tr_df[feats])
    X_te = scaler.transform(te_df[feats])
    y_tr = tr_df["maternal_anemia"].to_numpy()
    y_te = te_df["maternal_anemia"].to_numpy()

    # 3. Benchmark Dummy vs Logistic vs Overfit Tree
    dt4 = DecisionTreeClassifier(max_depth=4, random_state=seed)
    dt_full = DecisionTreeClassifier(random_state=seed)
    models = [
        ("Dummy (Majority)", DummyClassifier(strategy="most_frequent")),
        ("LogisticRegression", LogisticRegression(max_iter=500)),
        ("Tree (depth=4)", dt4),
        ("Tree (unpruned)", dt_full),
    ]
    records = []
    for name, mdl in models:
        mdl.fit(X_tr, y_tr)
        tr_acc = float(mdl.score(X_tr, y_tr))
        te_acc = float(mdl.score(X_te, y_te))
        records.append({
            "model": name, "train_acc": round(tr_acc, 4),
            "test_acc": round(te_acc, 4),
            "gap": round(tr_acc - te_acc, 4),
        })
    return pd.DataFrame(records)

print(run_leak_free_mch_benchmark(mch_raw).to_string(index=False))
